# ETL 02 — Extração das cotações da B3 (yfinance)

**Projeto:** Plataforma Financeira Integrada — Tríade (Linha 4)

**Fonte:** Histórico diário de cotações da B3 via API pública yfinance (sufixo `.SA`).

**Período:** 01/01/2019 a 31/12/2023 (período oficial da Linha 4).

**Carteira:** 9 companhias em 5 grupos setoriais, selecionadas a partir do universo sugerido no escopo do projeto.

**Saída:** CSV bruto em `data/raw/` (não versionado; reproduzível executando este notebook).

In [1]:
from pathlib import Path
import time
import pandas as pd
import yfinance as yf

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

print("Raiz do projeto:", ROOT)
print("Versão do yfinance:", yf.__version__)

Raiz do projeto: c:\Users\waldinei.rosa\Projetos\Projeto_Triade_Financeira_Waldinei
Versão do yfinance: 1.7.0


In [ ]:
CARTEIRA = {
    "LREN3": {"empresa": "Lojas Renner", "setor": "Consumo/Varejo"},
    "ABEV3": {"empresa": "Ambev", "setor": "Consumo/Varejo"},
    "PETR4": {"empresa": "Petrobras", "setor": "Energia"},
    "EGIE3": {"empresa": "Engie Brasil", "setor": "Energia"},
    "VALE3": {"empresa": "Vale", "setor": "Commodities Exportadoras"},
    "SUZB3": {"empresa": "Suzano", "setor": "Commodities Exportadoras"},
    "WEGE3": {"empresa": "WEG", "setor": "Exportadoras Industriais"},
    "EMBR3": {"empresa": "Embraer", "setor": "Exportadoras Industriais"},
    "ITUB4": {"empresa": "Itaú Unibanco", "setor": "Bancos"},
}

DATA_INICIAL = "2019-01-01"
DATA_FINAL_EXCLUSIVA = "2024-01-01"  # o yfinance NÃO inclui a data final; usar 2024-01-01 garante 31/12/2023
# Códigos alterados após o período analisado: o Yahoo Finance guarda o histórico no código atual
TICKER_YAHOO = {
    "EMBR3": "EMBJ3"
}  # Embraer passou a negociar como EMBJ3 na B3 em nov/2025

In [ ]:
def extrair_cotacoes(ticker, inicio, fim, tentativas=3):
    """Baixa o histórico diário de um ativo da B3 pelo yfinance e devolve um DataFrame."""
    simbolo = f"{TICKER_YAHOO.get(ticker, ticker)}.SA"
    for tentativa in range(1, tentativas + 1):
        try:
            df = yf.download(
                simbolo,
                start=inicio,
                end=fim,
                auto_adjust=False,
                progress=False,
                multi_level_index=False,
            )
            if df is not None and not df.empty:
                df = df.reset_index()
                df.insert(0, "ticker", ticker)
                return df
            print(f"  {simbolo}: retorno vazio (tentativa {tentativa})")
        except Exception as erro:  # noqa: BLE001 — o yfinance lança tipos variados de erro
            print(f"  {simbolo}: tentativa {tentativa} falhou ({erro})")
        time.sleep(3 * tentativa)
    raise RuntimeError(f"Não foi possível baixar {simbolo}.")

In [ ]:
frames, resumo = [], []
for ticker in CARTEIRA:
    df = extrair_cotacoes(ticker, DATA_INICIAL, DATA_FINAL_EXCLUSIVA)
    frames.append(df)
    resumo.append(
        {
            "ticker": ticker,
            "linhas": len(df),
            "primeira_data": df["Date"].min().date(),
            "ultima_data": df["Date"].max().date(),
            "nulos": int(df.isna().sum().sum()),
        }
    )
    time.sleep(1)  # pausa entre requisições: respeita os limites da API

cotacoes = pd.concat(frames, ignore_index=True)
cotacoes.to_csv(RAW_DIR / "b3_cotacoes_2019_2023.csv", index=False, encoding="utf-8")

print("Total de linhas:", len(cotacoes))
pd.DataFrame(resumo)

Total de linhas: 11169


,ticker,linhas,primeira_data,ultima_data,nulos
0,LREN3,1241,2019-01-02,2023-12-28,0
1,ABEV3,1241,2019-01-02,2023-12-28,0
2,PETR4,1241,2019-01-02,2023-12-28,0
3,EGIE3,1241,2019-01-02,2023-12-28,0
4,VALE3,1241,2019-01-02,2023-12-28,0
5,SUZB3,1241,2019-01-02,2023-12-28,0
6,WEGE3,1241,2019-01-02,2023-12-28,0
7,EMBR3,1241,2019-01-02,2023-12-28,0
8,ITUB4,1241,2019-01-02,2023-12-28,0


In [5]:
print(cotacoes.dtypes)
cotacoes.head()

ticker                 str
Date         datetime64[s]
Adj Close          float64
Close              float64
High               float64
Low                float64
Open               float64
Volume               int64
dtype: object


,ticker,Date,Adj Close,Close,High,Low,Open,Volume
0,LREN3,2019-01-02,25.094881,32.460915,32.729248,31.305590,31.305590,5471099
1,LREN3,2019-01-03,24.777952,32.050961,32.460915,31.700638,32.460915,4668412
2,LREN3,2019-01-04,23.827168,30.821098,32.095684,30.672026,32.095684,7069631
3,LREN3,2019-01-07,23.279745,30.112995,31.096886,29.859570,30.821098,5270125
4,LREN3,2019-01-08,24.028856,31.081980,31.141609,30.083181,30.358969,7234247


## Conferência da extração

| Verificação | Resultado |
|---|---|
| Ativos extraídos | 9 de 9 |
| Pregões por ativo | 1.241 (contagem idêntica para todos os ativos) |
| Total de registros | 11.169 (9 × 1.241) |
| Período efetivo | 02/01/2019 (primeiro pregão) a 28/12/2023 (último pregão de 2023) |
| Valores nulos | 0 |

**Observações:**
- A B3 tem calendário próprio de pregões, que difere dos dias úteis da PTAX (BACEN). A integração entre as fontes deverá considerar essa diferença.
- A Embraer teve o código de negociação alterado de **EMBR3 para EMBJ3** em 2025. O projeto mantém o código histórico (EMBR3), vigente no período analisado, e busca o histórico pelo código atual no Yahoo Finance.
- `Close` (preço nominal) será usado no cálculo do Volume Financeiro; `Adj Close` (ajustado por proventos) será usado nos retornos e na Base 100.